In [1]:
# ==========================================
# PHASE 5.3 - FUTURE CLV DATASET CREATION
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# Load Data
# ------------------------------------------

df = pd.read_csv("../data/cleaned_online_retail.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Dataset Shape:", df.shape)

# ------------------------------------------
# Date Range
# ------------------------------------------

print("\nMin Date:", df["InvoiceDate"].min())
print("Max Date:", df["InvoiceDate"].max())

# ------------------------------------------
# Split Date
# ------------------------------------------

split_date = pd.Timestamp("2011-09-01")

past_data = df[df["InvoiceDate"] < split_date].copy()
future_data = df[df["InvoiceDate"] >= split_date].copy()

print("\nPast Transactions:", past_data.shape)
print("Future Transactions:", future_data.shape)

# ------------------------------------------
# Snapshot Date
# ------------------------------------------

snapshot_date = split_date

# ------------------------------------------
# Customer Features (Past Period)
# ------------------------------------------

features = past_data.groupby("CustomerID").agg({

    "InvoiceDate": [
        lambda x: (snapshot_date - x.max()).days,
        "min",
        "max"
    ],

    "InvoiceNo": "nunique",

    "Revenue": [
        "sum",
        "mean"
    ],

    "Quantity": [
        "sum",
        "mean"
    ]

})

features.columns = [

    "Recency",
    "FirstPurchase",
    "LastPurchase",

    "Frequency",

    "PastRevenue",
    "AvgOrderValue",

    "TotalQuantity",
    "AvgQuantity"

]

features = features.reset_index()

# ------------------------------------------
# Active Days
# ------------------------------------------

features["ActiveDays"] = (

    features["LastPurchase"] -
    features["FirstPurchase"]

).dt.days

features.drop(
    columns=[
        "FirstPurchase",
        "LastPurchase"
    ],
    inplace=True
)

# ------------------------------------------
# Future Revenue Target
# ------------------------------------------

future_clv = (

    future_data.groupby("CustomerID")["Revenue"]
    .sum()
    .reset_index()

)

future_clv.columns = [

    "CustomerID",
    "FutureRevenue"

]

# ------------------------------------------
# Merge
# ------------------------------------------

clv_df = pd.merge(

    features,
    future_clv,

    on="CustomerID",

    how="left"

)

# Customers with no future purchases

clv_df["FutureRevenue"] = (
    clv_df["FutureRevenue"]
    .fillna(0)
)

# ------------------------------------------
# Log Target
# ------------------------------------------

clv_df["FutureRevenue_Log"] = np.log1p(
    clv_df["FutureRevenue"]
)

print("\nFinal Dataset Shape:")
print(clv_df.shape)

print("\nFirst 5 Rows:")
print(clv_df.head())

# ------------------------------------------
# Save
# ------------------------------------------

clv_df.to_csv(
    "../data/future_clv_dataset.csv",
    index=False
)

print("\nFuture CLV Dataset Saved!")

Dataset Shape: (392692, 9)

Min Date: 2010-12-01 08:26:00
Max Date: 2011-12-09 12:50:00

Past Transactions: (224036, 9)
Future Transactions: (168656, 9)

Final Dataset Shape:
(3317, 10)

First 5 Rows:
   CustomerID  Recency  Frequency  PastRevenue  AvgOrderValue  TotalQuantity  \
0     12346.0      225          1     77183.60   77183.600000          74215   
1     12347.0       29          5      2790.86      22.506935           1590   
2     12348.0      148          3      1487.24      53.115714           2124   
3     12350.0      210          1       334.40      19.670588            197   
4     12352.0      162          5      1561.81      41.100263            254   

    AvgQuantity  ActiveDays  FutureRevenue  FutureRevenue_Log  
0  74215.000000           0           0.00           0.000000  
1     12.822581         237        1519.14           7.326558  
2     75.857143         109         310.00           5.739793  
3     11.588235           0           0.00           0.000000 